# Missing imputation

- `owid.csv`: OWID dataset in which each corresponds to a row. Before importing to this notebook, we have already done the following preprocessing steps manually:

  - Only keep 12 columns, as mentioned in the manuscript.

  - Only keep 232 countries entering the survival analysis.

  - Only keep the data within the first entery date until the event or censoring date for each country.

  - Manually collect data from trusthworthy sources to fill in some missing values.

In [1]:
import pandas as pd
import warnings

from missingpy import MissForest

warnings.filterwarnings("ignore")

file_path = "../data/owid.csv"

original_df = pd.read_csv(file_path)
original_df.drop(columns=['population_density'], inplace=True)

print(f"Number of countries: {original_df['iso_code'].nunique()}")

original_df.info()

Number of countries: 232
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156613 entries, 0 to 156612
Data columns (total 13 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   iso_code                             156613 non-null  object 
 1   continent                            156613 non-null  object 
 2   date                                 156613 non-null  object 
 3   new_cases_smoothed_per_million       156613 non-null  float64
 4   median_age                           156613 non-null  float64
 5   population                           156613 non-null  float64
 6   classification                       156613 non-null  object 
 7   group                                156613 non-null  object 
 8   location                             156613 non-null  object 
 9   total_tests_per_thousand             41618 non-null   float64
 10  people_fully_vaccinated_per_hundred  13518 non-null   f

## Imputation using MissForest

- 3 target (missed) columns:

    + `total_tests_per_thousand`: total tests per thousand people.

    + `people_vaccinated_per_hundred`: people vaccinated per hundred.

    + `stringency_index`: government response stringency index.

In [16]:
cat_cols = original_df.select_dtypes(include=["object"]).columns
num_cols = original_df.select_dtypes(include=["float64", "int64"]).columns

original_df[cat_cols] = original_df[cat_cols].astype("category")

cat_mappings = {col: original_df[col].cat.categories for col in cat_cols}
original_df[cat_cols] = original_df[cat_cols].apply(lambda x: x.cat.codes)

imputer = MissForest(random_state=42)
imputed_array = imputer.fit_transform(original_df)

# Convert back to DataFrame
original_df = pd.DataFrame(imputed_array, columns=original_df.columns)

# Recover categorical dtypes
for col in cat_cols:
    original_df[col] = original_df[col].round().astype(int)  # MissForest returns floats
    original_df[col] = pd.Categorical.from_codes(original_df[col], categories=cat_mappings[col])

original_df.info()

Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156613 entries, 0 to 156612
Data columns (total 13 columns):
 #   Column                               Non-Null Count   Dtype   
---  ------                               --------------   -----   
 0   iso_code                             156613 non-null  category
 1   continent                            156613 non-null  category
 2   date                                 156613 non-null  category
 3   new_cases_smoothed_per_million       156613 non-null  float64 
 4   median_age                           156613 non-null  float64 
 5   population                           156613 non-null  float64 
 6   classification                       156613 non-null  category
 7   group                                156613 non-null  category
 8   location                             156613 non-null  category
 9   total_tests_per_thousand             156613 non-null  float64 
 10  peo

## Data labeling

- Create a new column `status`:

    + `status = True` if `new_cases_smoothed_per_million` >= 250

    + `status = False` if `new_cases_smoothed_per_million` < 250

- Create a new column `days`: the number of days from the first entery date until the event or censoring date for each country.

In [17]:
original_df['days'] = original_df.groupby('iso_code').cumcount() + 1
original_df['status'] = original_df['new_cases_smoothed_per_million'] >= 250

original_df.info()

print(f"--------------------------------")
print(f"Number of events: {original_df[original_df['status'] == True].shape[0]}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156613 entries, 0 to 156612
Data columns (total 15 columns):
 #   Column                               Non-Null Count   Dtype   
---  ------                               --------------   -----   
 0   iso_code                             156613 non-null  category
 1   continent                            156613 non-null  category
 2   date                                 156613 non-null  category
 3   new_cases_smoothed_per_million       156613 non-null  float64 
 4   median_age                           156613 non-null  float64 
 5   population                           156613 non-null  float64 
 6   classification                       156613 non-null  category
 7   group                                156613 non-null  category
 8   location                             156613 non-null  category
 9   total_tests_per_thousand             156613 non-null  float64 
 10  people_fully_vaccinated_per_hundred  156613 non-null  float64 
 11  

## File splitting

- Split the dataset into 2 files:

    + `time depdendent variables`

    + `time independent variables`

In [25]:
# Split into time depedent and time independent variables
tdp_df = original_df[[
    'iso_code', 'total_tests_per_thousand', 'people_fully_vaccinated_per_hundred', 'stringency_index', 'days', 'status'
]].copy()

idp_df = original_df[[
    'iso_code', 'continent', 'population', 'location', 'median_age',
    'group', 'classification', 'Overall_score', 'days', 'date', 'status'
]].copy()

tdp_df.info()

# Grouping by 'iso_code'
# Aggregation logic:
## 'date' --> 'first_date': takes the first date for each country
## 'days' --> 'duration': takes the maximum days for each country
## 'status' --> 'status': takes the maximum status for each country (if any event occurred, it will be True)

idp_df = idp_df.groupby('iso_code').agg({
    'continent': 'first',
    'population': 'first',
    'location': 'first',
    'median_age': 'first',
    'classification': 'first',
    'Overall_score': 'first',
    'group': 'first',
    'date': 'first',
    'days': 'max',
    'status': 'max'
}).reset_index()

idp_df.rename(columns={'days': 'duration', 'date': 'first_case'}, inplace=True)

idp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156613 entries, 0 to 156612
Data columns (total 6 columns):
 #   Column                               Non-Null Count   Dtype   
---  ------                               --------------   -----   
 0   iso_code                             156613 non-null  category
 1   total_tests_per_thousand             156613 non-null  float64 
 2   people_fully_vaccinated_per_hundred  156613 non-null  float64 
 3   stringency_index                     156613 non-null  float64 
 4   days                                 156613 non-null  int64   
 5   status                               156613 non-null  bool    
dtypes: bool(1), category(1), float64(3), int64(1)
memory usage: 5.2 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232 entries, 0 to 231
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   iso_code        232 non-null    category
 1   continent       232 non-nu

## Store files

In [26]:
idp_path = "../data/time-independent-variables-MissForest.csv"
tdp_path = "../data/time-dependent-variables-MissForest.csv"

idp_df.to_csv(idp_path, index=False)
tdp_df.to_csv(tdp_path, index=False)